In [ ]:
# 0. 라이브러리
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score, roc_auc_score,
                             classification_report, confusion_matrix)

RS = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [ ]:
# 1. 데이터 불러오기 + 팀 공통 Train/Test 분할 (index 파일 기준, 80:20 층화)
df = pd.read_csv(ROOT / "data/processed/Faults_processed.csv")      # 원본 26개 피처 + target (A400 제외된 상태)
train_idx = pd.read_csv(ROOT / "data/split/train_idx.csv")["index"]
test_idx = pd.read_csv(ROOT / "data/split/test_idx.csv")["index"]

X, y = df.drop(columns="target"), df["target"]
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print("Train", X_train.shape, "| Test", X_test.shape)

Train (1552, 26) | Test (389, 26)


In [ ]:
# 2. 파생변수 10개 생성
PLATE_KEYS = ["Length_of_Conveyer", "Steel_Plate_Thickness", "TypeOfSteel_A300"]   # 강판 조건
DISCRETE = ["TypeOfSteel_A300", "Outside_Global_Index"]                             # 이산형 변수 (마지막 열로 배치)

def plate_key(X):
    return X[PLATE_KEYS].astype(str).agg("_".join, axis=1)

def add_features(X, plate_counts):
    out = X.copy()
    width = (X["X_Maximum"] - X["X_Minimum"]).clip(lower=0)
    mx, mn = X["Maximum_of_Luminosity"], X["Minimum_of_Luminosity"]
    # 형상
    out["compactness"] = (X["X_Perimeter"] + X["Y_Perimeter"]) / np.sqrt(X["Pixels_Areas"])   # 둘레 대비 면적
    out["perimeter_ratio_log"] = np.log((X["X_Perimeter"] + 1) / (X["Y_Perimeter"] + 1))      # 가로/세로 둘레 비
    out["x_straightness"] = X["X_Perimeter"] / (width + 1)                                     # 가로 윤곽의 굴곡
    # 밝기
    out["lum_range"] = mx - mn                                                                 # 밝기 범위
    out["lum_contrast"] = (mx - mn) / (mx + mn).replace(0, 1)                                  # 상대 대비
    out["lum_mean_position"] = (X["Sum_of_Luminosity"] / X["Pixels_Areas"] - mn) / (mx - mn + 1)  # 평균 밝기의 위치
    # 위치
    xc = (X["X_Minimum"] + X["X_Maximum"]) / 2 / X["Length_of_Conveyer"]
    out["x_center_rel"] = xc                                                                   # 컨베이어 폭 대비 중심 위치
    out["x_center_dist"] = (xc - 0.5).abs()                                                    # 중앙에서 떨어진 정도
    # 강판 맥락
    out["plate_defect_count"] = plate_key(X).map(plate_counts).fillna(0).astype(float)        # 같은 강판 조건의 결함 수 (처음 보는 조건 = 0)
    out["thickness_per_conveyer"] = X["Steel_Plate_Thickness"] / X["Length_of_Conveyer"]
    # 연속형 → 이산형 순서로 열 정렬 (학습·평가 열 순서 고정)
    return out[[c for c in out.columns if c not in DISCRETE] + DISCRETE]

In [ ]:
# 3. 최종 모델: ExtraTrees + Random Forest + GBM Soft Voting (36개 피처에서 튜닝한 하이퍼파라미터)
def fit_soft_voting(X_tr, y_tr, n_jobs=-1):
    plate_counts = plate_key(X_tr).value_counts().to_dict()      # 강판 조건별 결함 수는 학습 데이터로만 계산 (누수 방지)
    Xf = add_features(X_tr, plate_counts)
    et = ExtraTreesClassifier(n_estimators=438, max_depth=15, max_features=0.7, min_samples_leaf=1,
                              class_weight="balanced", random_state=RS, n_jobs=n_jobs)     # 불균형: 클래스 가중치
    rf = RandomForestClassifier(n_estimators=483, max_depth=None, max_features=0.3, min_samples_leaf=1,
                                random_state=RS, n_jobs=n_jobs)                             # 불균형 처리 없음
    gbm = GradientBoostingClassifier(n_estimators=240, learning_rate=0.08793919399591556, max_depth=4,
                                     subsample=0.9079017872741743, max_features=0.5, min_samples_leaf=5,
                                     random_state=RS)
    # GBM은 class_weight 인자가 없으므로 균형 가중치를 sample_weight로 전달: w_c = 최다 클래스 건수 / 클래스 c 건수
    counts = y_tr.value_counts()
    et.fit(Xf, y_tr); rf.fit(Xf, y_tr); gbm.fit(Xf, y_tr, sample_weight=y_tr.map(counts.max() / counts).to_numpy())
    return [et, rf, gbm], plate_counts

def predict_soft_voting(models, plate_counts, X_new):
    Xf = add_features(X_new, plate_counts)
    proba = np.mean([m.predict_proba(Xf) for m in models], axis=0)   # 세 모델의 클래스별 확률 평균
    pred = models[0].classes_[proba.argmax(axis=1)]                   # 확률이 가장 높은 클래스
    return pred, proba

def metrics(y_true, pred, proba, classes):
    return {"Macro F1": f1_score(y_true, pred, average="macro"),
            "Accuracy": accuracy_score(y_true, pred),
            "Macro Precision": precision_score(y_true, pred, average="macro"),
            "Macro Recall": recall_score(y_true, pred, average="macro"),
            "Weighted F1": f1_score(y_true, pred, average="weighted"),
            "Macro ROC-AUC": roc_auc_score(y_true, proba, multi_class="ovr", average="macro", labels=classes)}

In [ ]:
# 4. Train 내부 반복 층화 CV (5-fold × 3회 = 15 fold) — 모델 선택 기준 성능
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RS)

def eval_fold(tr, va):
    models, pc = fit_soft_voting(X_train.iloc[tr], y_train.iloc[tr], n_jobs=1)
    pred, proba = predict_soft_voting(models, pc, X_train.iloc[va])
    return metrics(y_train.iloc[va], pred, proba, models[0].classes_)

cv_scores = pd.DataFrame(Parallel(n_jobs=-1)(delayed(eval_fold)(tr, va) for tr, va in rcv.split(X_train, y_train)))   # 몇 분 소요
cv_scores.index = [f"fold {i + 1}" for i in range(len(cv_scores))]
print(f"반복 CV(5×3) Macro F1 : {cv_scores['Macro F1'].mean():.4f} ± {cv_scores['Macro F1'].std(ddof=0):.4f}"
      f"   (0.83 이상 fold {(cv_scores['Macro F1'] >= 0.83).sum()}/15)")
cv_summary = pd.DataFrame({"평균": cv_scores.mean(), "표준편차": cv_scores.std(ddof=0)}).round(4)   # 지표별 15 fold 평균 ± 표준편차
cv_summary

반복 CV(5×3) Macro F1 : 0.8389 ± 0.0231   (0.83 이상 fold 10/15)


,평균,표준편차
Macro F1,0.8389,0.0231
Accuracy,0.8127,0.0208
Macro Precision,0.8562,0.0266
Macro Recall,0.8272,0.0262
Weighted F1,0.8127,0.0207
Macro ROC-AUC,0.9706,0.0053


In [ ]:
# 5. Train 전체로 학습 → Test 예측
models, plate_counts = fit_soft_voting(X_train, y_train)
pred, proba = predict_soft_voting(models, plate_counts, X_test)

In [ ]:
# 6. Test 성능
order = ["Other_Faults", "Bumps", "K_Scratch", "Z_Scratch", "Pastry", "Stains", "Dirtiness"]
test_scores = pd.Series(metrics(y_test, pred, proba, models[0].classes_)).round(4)
print("Test 성능"); print(test_scores.to_string(), "\n")
print(classification_report(y_test, pred, labels=order, digits=3))
pd.DataFrame(confusion_matrix(y_test, pred, labels=order), index=order, columns=order)   # 행 = 실제, 열 = 예측

Test 성능
Macro F1           0.8324
Accuracy           0.8098
Macro Precision    0.8486
Macro Recall       0.8182
Weighted F1        0.8123
Macro ROC-AUC      0.9686 

              precision    recall  f1-score   support

Other_Faults      0.745     0.800     0.771       135
       Bumps      0.756     0.728     0.742        81
   K_Scratch      0.986     0.923     0.954        78
   Z_Scratch      1.000     0.947     0.973        38
      Pastry      0.529     0.562     0.545        32
      Stains      0.923     0.857     0.889        14
   Dirtiness      1.000     0.909     0.952        11

    accuracy                          0.810       389
   macro avg      0.849     0.818     0.832       389
weighted avg      0.816     0.810     0.812       389



,Other_Faults,Bumps,K_Scratch,Z_Scratch,Pastry,Stains,Dirtiness
Other_Faults,108,14,1,0,11,1,0
Bumps,18,59,0,0,4,0,0
K_Scratch,5,0,72,0,1,0,0
Z_Scratch,2,0,0,36,0,0,0
Pastry,10,4,0,0,18,0,0
Stains,1,1,0,0,0,12,0
Dirtiness,1,0,0,0,0,0,10


In [ ]:
# 7. 모델 저장 — 세 모델 + 강판 조건별 결함 수(Train 기준)를 함께 저장 (예측 시 파생변수 계산에 필요)
MODEL_PATH = ROOT / "models" / "Taegyeom_final_model_trained.pkl"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
bundle = {"models": models,                      # [ExtraTrees, RandomForest, GBM]
          "plate_counts": plate_counts,          # plate_defect_count 계산용
          "feature_columns": list(add_features(X_train.head(1), plate_counts).columns),   # 학습 시 열 순서 (36개)
          "test_macro_f1": round(f1_score(y_test, pred, average="macro"), 4)}
joblib.dump(bundle, MODEL_PATH, compress=3)      # 압축 저장 (약 16MB)
print("저장:", MODEL_PATH)

저장: c:\Users\tae37\Desktop\workspace1\steel-plate-fault-classification\models\Taegyeom_final_model_trained.pkl
